## Plotting spatial activity

In this notebook, you will plot topographies of scalp EEG. 
EEG topographies resemble maps that present the spatial distribution of EEG activity across the scalp-level electrodes. We can refer to it as a **topography** or a **topographical map**. A topographical representation of activity allows us to see over which regions (**where**) activity is concentrated. For example, a **red** region indicates a concentration of positive activity over that area, which we can relate to **excitatory** activity, while a **blue** region is suggestive of **inhibitory** activity. 

But, these concepts of **excitation** and **inhibition** make more physical sense, if we represent our scalp EEG as **Current Source Density (CSD)** or the *underlying flow of current* rather than simply as potential differences recorded at the scalp. Representing scalp EEG as CSD gives us an insight into the current **sources** and **sinks**, where **sources** correspond to positive extracellular activity and **sinks** refer to negative extracellular activity. 

<div class="alert alert-block alert-info"> 
An electrode recording extracellular <b>local field potentials</b> would record <b>positive activity</b> in the case where there has been <b>excitatory post-synaptic potentials</b> (excitatory PSPs) close to the recording site and <b>negative activity</b> in the case of **inhibitory post-synaptic potentials**. 

At rest the intracellular environment of a neuron is negatively polarized at approximately -70 mV compared with the extracellular environment. The potential difference is due to an unequal distribution of Na+, K+ and Cl- ions across the cell membrane. This unequal distribution is maintained by the Na+ and K+ ion pumps located in the cell membrane. 
Excitatory PSPs occur at the level of a neuron's dendrites when a neurotransmitter (Glutamate for example), secreted by an active pre-synaptic neuron, makes the post-synaptic neuron's membrane more permeable to positively charged ions (cations) like Sodium (Na+). When these positively charged particulars flow into the intracellular space of the neuron this leads to a **depolarisation** of the neuron. This depolarisation effectively activates the neuron, increasing the probability of an action potential. This also leads to a local extracellular **sink**.

Inversely, Inhibitory PSPs (IPSP), which also occur at the level of a neuron's dendrites, occur when a neurotransmitter (for example, GABA) provokes the opening of ion channels sensitive to negatively charged ions (anions) such as Chloride (Cl-). When these negatively charged ions cross the neural membrane into the intracellular space, this leads to a **hyperpolarisation** of the neuron. The neuron is more negatively charged inside compared to outside and this hyperpolarisation effectively leads to an inhibition of the neuron, decreasing the probability of an action potential. This also gives rise to an extracellular **source**. 
However, an **extracellular sink** always needs to be balance by an **extracellular source**, this is necessary to maintain electroneurality. Therefore, the influx of cations into the neuron is always balanced by an opposing flux of negatively charged ions from the intracellular space to the extracellular space. This oposite flow of ions is referred to as the **return current** or the **passive current**. Depending on the distance of the sink from its source, a dipole is formed and thus, an electrical current (A dipole is formed by two opposing charges that are seperated by a small distance and is characterised by its orientation and its position).  

</div>


The EEG that we record at the level of the scalp is spatio-temporally smoothed or *smeared** version of local field potentials. This smoothing or smearing is due to the phenomenon of **volume conduction**, which is a distortion and attenuation of the activity as it passes through the soft and hard tissues that seperate the source of the current, from the recording electrode at the level of the head. Current Source Density (CSD) techniques, techniques that help to represent the scalp-level activity as the underlying current flow, reduce the effects of volume conduction allowing us to distinguish better the activity from one region to the next. This is possible as they apply a mathematical method called the **Surface Laplacian**. 

<div class="alert alert-block alert-success"> 
Remember also that it is hypothesized that it is the result of a summation of the post-synaptic potentials from a large number of neurons that contributes most to the activity that we record at the level of the scalp. Action potential activity, because of its very short duration, is thought to be less likely to summate and, therefore, contribute to scalp EEG. 
</div>

In the first step, we load in the dataset and carry out some basic preprocessing steps :
- Define the type of each of the electrodes to distinguish the 64 scalp electrodes from the external electrodes that record eye-blinks (EOG- electro-oculogramme) for example.
- High pass filter the data so that all frequencies below 0.01Hz are filtered out.
- Re-reference the data of all electrodes to the "average" reference. This means that we use the average over all our scalp electrodes as the reference. 

In [ ]:
import mne
import numpy as np
import matplotlib.pyplot as plt
import os

# We begin by loading in .fif file that we saved at the end of script 1.
# This is the data that has been high-pass filtered and re-referenced. 
fnameIn = 'sub-001_eeg_sub-001_task-med2_eeg_short.set'
fpathIn = 'Data'
fullnameIn = os.path.join(fpathIn, fnameIn)
rawIn = mne.io.read_raw_eeglab(fullnameIn, preload=True)

my_dict = {'EXG1': 'eog', 'EXG2': 'eog', 'EXG3': 'eog', 'EXG4': 'eog', 'EXG5': 'eog', 'EXG6': 'eog', 'EXG7': 'eog',
           'EXG8': 'eog',
           'GSR1': 'misc', 'GSR2': 'misc', 'Erg1': 'stim', 'Erg2': 'stim', 'Resp': 'resp', 'Plet': 'misc',
           'Temp': 'misc'}
print(my_dict)
rawIn.set_channel_types(my_dict)  # Apply the channel type to our raw object.

# Highpass filter
rawInFilt = rawIn.copy().filter(0.1, None,
                                fir_design='firwin') 

# Rereference to the average reference.
rawInRef = rawInFilt.copy().pick_types(eeg=True, exclude=['bads', 'eog']).set_eeg_reference(ref_channels='average')
mne.viz.plot_raw(rawInRef, scalings='auto', remove_dc=False)

## Plot topographies over a defined interval

When we want to look at the spatial activity of EEG, we have to calculate the mean activity at each scalp electrode over a specified time interval.  
A topography over a specified time interval can be used to highlight activity at a specific time window. 
To plot a single topography, we need to define a vector of the mean activity over the defined time interval for each
electrode.
Before we can visualize the topography, we need to specify the electrode layout that we are using or "montage"; this electrode layout should correspond to the electrode layout used when recording the EEG data. 

In the following example, we tell MNE-Python to use the standard **10-20** layout. We also visualise the standard layout to make sure that it is correct. We then define as our time interval of interval of interest the interval between 5seconds and 10seconds. The **_time_as_index()** method allows us to find the samples that correspond to the 5second and 10second timepoints. We then extract the data over this time interval for the 64 scalp electrodes and calculate the mean over this time interval for each electrode. Then we plot. 

In [ ]:
montage = mne.channels.make_standard_montage('standard_1020')  # Assigning the standard 10-20 montage
mne.viz.plot_montage(mne.channels.make_standard_montage('standard_1020'))  # Visualize the montage
rawInRef.set_montage(montage)

timeIntval = [40, 45]  # Defining the time interval over which to plot the topography.
timeIndx = rawInRef.time_as_index(timeIntval)  # Find the indices of the samples in the defined time interval.
chanRange = np.arange(0, 64)
dataSeg1 = rawInRef.get_data(chanRange, timeIndx[0], timeIndx[1])
dataSeg_mean = np.mean(dataSeg1, 1)

fig1, ax1 = plt.subplots(1)
mne.viz.plot_topomap(dataSeg_mean, rawInRef.info, ch_type='eeg', axes=ax1)


Now we will plot several topographies over time from 60seconds to 80seconds in 5 second steps.
This allows us to appreciate the change in activity across all electrodes over time

In [ ]:
timeIntvals = np.arange(5, 25, 5)
fig2, axes = plt.subplots(1, len(timeIntvals) - 1, figsize=(15, 5))
for ind in range(len(timeIntvals) - 1):
    ind
    curr_int = [timeIntvals[ind], timeIntvals[ind + 1]]
    timeIndx2 = rawInRef.time_as_index(curr_int)
    dataSegCurr = rawInRef.get_data(chanRange, timeIndx2[0], timeIndx2[1])
    dataMeanCurr = np.mean(dataSegCurr, 1)

    mne.viz.plot_topomap(dataMeanCurr, rawInRef.info, ch_type='eeg', axes=axes[ind], show = False)
    axes[ind].set_title(str(timeIntvals[ind]) + ' - ' + str(timeIntvals[ind + 1]) + 'seconds', {'fontsize': 20})
    
plt.show()

## Estimate the Current Source Density by calculating the Surface Laplacian

The aim of estimating the **Current Source Density** (CSD) via the **Spatial Laplacian** is to reduce the "smearing" effects due to volume conduction and to have a topography that differentiates more clearly the activity between different regions. Also, as the CSD reflects the flow of the underlying currents generating the scalp-level EEG, it gives us a better insight into the underlying neural activity. 

The key parameters of the spatial Laplacian are:

the m-constant that affects the stiffness or rigidity of the spherical spline. This should ideally be set to 4.
The smoothing constant, *lambda* is set to .000001 by default.
You can try out different values of these parameters and see if/how they alter the resulting topography.

It is important to have added the channel montage to the dataset before carrying out the spatial Laplacian.

What difference, if any, do you notice in the topography after applying the Spatial Laplacian compared to before? 


In [ ]:
# Estimate the current source density using the Laplacian 

rawCSD = mne.preprocessing.compute_current_source_density(rawInRef, stiffness=4, lambda2=1e-5)
data2plot_csd = np.array(rawCSD.get_data())

data2plot_csd_seg = data2plot_csd[chanRange, timeIndx[0]:timeIndx[1]]
data2plot_csd_segmean = np.mean(data2plot_csd_seg,1)

# Visualize the average  CSD activity over the time interval as a topography
fig2, ax2 = plt.subplots(1)
mne.viz.plot_topomap(data2plot_csd_segmean, rawCSD.info, ch_type='eeg', axes=ax2)

## Your turn !

Try plotting the topography of activity over different time windows of the data. Just make sure that you now the duration of the data before-hand; there is no point trying to plot between 50seconds and 60seconds if the dataset only lasts 30seconds.
Then apply the Surface Laplacian and see the difference...

Try also plotting the temporal signal of the EEG before and after applying the Surface Laplacian.
*Remember that the code for visualing the temporal signals is in "DSC_2026_script1_basics.ipynb"*